&nbsp;

## ChIP-Atlas Diff Analysis Parse Notebook

- 既に `diff Analysis` を実行済みだった heat-stress related ATAC-seq dataset に対し、`bedtools intersect` を行い、その結果をparse するための notebook

In [1]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl

from IPython.display import display


os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### この notebook を実行するために必要な手順

1. [ChIP-Atlas Diff Analysis](https://chip-atlas.org/diff_analysis) を実行する
    1. 事前に定義した heat-stress (dataset A) vs. ctrl (dataset B) のデータに従ってペアを作成し、実行する
2. `bedtools intersect -wo`を実行する CWLファイル [../../Workflow/bedtools_intersect_v2_scatter.cwl](../../Workflow/bedtools_intersect_v2_scatter.cwl)を実行する

&nbsp;

```bash
cwltool --outdir ./out/ChIP-Atlas_diff/ATACseq/bedfile_intersect_v2/ \
./Workflow/bedtools_intersect_v2_scatter.cwl \
./config/bedtools_intersect/chipatlas_diff_atacseq_v2.yml
```

&nbsp;

#### WARNING

- 以下の処理では、q-value < 0.01 でフィルタリングしている

In [2]:
INTERSECT_DIFF_COLUMNS = [
    # A: ChIP-Atlas diff DPR (8 cols)  <- query (-a)
    "dpr_chrom", "dpr_start", "dpr_end",
    "norm_count_A", "norm_count_B",  # dataset A → Heat stress, B → Control
    "log2fc", "p_value", "q_value",
    # B: fanta.bio CRE (11 cols)       <- database (-b)
    "cre_chrom", "cre_start", "cre_end", "cre_id",
    "cre_score", "cre_strand", "thickStart", "thickEnd",
    "itemRgb", "cre_name", "pooled_cre_scores",
    # -wo
    "overlap_bp",
]

In [3]:
#繰り返し処理を行う
BED_INTERSECT_DIR = pathlib.Path("../../out/ChIP-Atlas_diff/ATACseq/bedfile_intersect_v2")

def read_intersect_bed(bed_path: pathlib.Path, q_threshold: float = 0.01) -> pl.DataFrame:
    """
    Read one bedtools intersect -wo output and tag with diff dataset ID from filename.
    default q-value threshold is 0.01.
    """
    # v2 filename: <dataset>_intersecting_<cre-peaks>.bed
    diff_dataset = bed_path.stem.split("_intersecting_", 1)[0]

    return (
        pl.read_csv(
            bed_path,
            separator="\t",
            has_header=False,
            new_columns=INTERSECT_DIFF_COLUMNS,
            # Force float dtypes: leading "1" q-values can make Polars infer Int64
            # and drop scientific-notation significant hits under ignore_errors.
            schema_overrides={
                "dpr_start": pl.Int64,
                "dpr_end": pl.Int64,
                "log2fc": pl.Float64,
                "p_value": pl.Float64,
                "q_value": pl.Float64,
                "cre_start": pl.Int64,
                "cre_end": pl.Int64,
                "cre_score": pl.Int64,
                "thickStart": pl.Int64,
                "thickEnd": pl.Int64,
                "overlap_bp": pl.Int64,
            },
            n_threads=16,
            ignore_errors=True,
        )
        .filter(pl.col("q_value") < q_threshold)
        .with_columns(
            # Fraction of CRE length covered by the DPR overlap (-wo)
            (pl.col("overlap_bp") / (pl.col("cre_end") - pl.col("cre_start"))).alias("overlap_frac_of_fantabio_cre"),
            pl.struct(
                "dpr_start",
                "dpr_end",
                "dpr_chrom"
            ).alias("dpr_interval")
        )
        .with_columns(
            pl.col("pooled_cre_scores").str.extract(r"directionality:([^|]+)", 1).cast(pl.Float64, strict=False).alias("directionality"),
            pl.col("pooled_cre_scores").str.extract(r"class:([^|]+)", 1).alias("fantabio_cre_class"),
        )
        .with_columns(
            pl.lit(diff_dataset).alias("diff_dataset"),
            pl.lit(bed_path.name).alias("source_bed_file"),
        )
        .drop("cre_score", "cre_strand", "thickStart", "thickEnd", "itemRgb", "pooled_cre_scores", "norm_count_A", "norm_count_B", "dpr_start", "dpr_end", "dpr_chrom")
    )

bed_paths = sorted(
    BED_INTERSECT_DIR.glob("*_intersecting_*.bed")
)

all_diff_df = (
    pl.concat(
        [read_intersect_bed(p) for p in bed_paths],
        how="vertical_relaxed",
    )
    .sort(
        "cre_chrom",
        descending=False,
    )
)


# mismatch = all_diff_df.filter(pl.col("cre_chrom") != pl.col("dpr_chrom"))
# print(f"mismatch rows: {mismatch.height}")
pl.Config.set_tbl_rows(10)
pl.Config.set_fmt_str_lengths(100)
display(all_diff_df)

log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str
1.346413,5.4291e-53,7.6313e-50,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",12,0.0625,"{629141,629153,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.363021,2.5793e-49,3.1990e-46,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",88,0.458333,"{629153,629241,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.520449,1.7373e-20,4.2758e-18,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",22,0.114583,"{629241,629263,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.470083,4.9866e-9,2.4993e-7,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",5,0.026042,"{629263,629268,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.510948,3.2981e-17,6.1358e-15,"""chr1""",629081,629273,"""FCHS_5""","""cp2@MTND1P23""",5,0.026042,"{629268,629276,""chr1""}",1.0,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.782456,0.000625,0.005354,"""chrY""",26670966,26671518,"""FCHS_447308""","""FCHS_447308""",184,0.333333,"{26670970,26671154,""chrY""}",-0.979798,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
0.966208,0.000259,0.004473,"""chrY""",11329762,11329941,"""FCHS_447044""","""FCHS_447044""",73,0.407821,"{11329868,11330053,""chrY""}",-0.884058,"""ELA""","""GSE101731_atac_hff_44C_60min""","""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.249991,0.000216,0.003874,"""chrY""",19077163,19077775,"""FCHS_447246""","""cp8@TTTY14""",99,0.161765,"{19077100,19077262,""chrY""}",-0.998313,"""PLA""","""GSE101731_atac_hff_44C_60min""","""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


&nbsp;

&nbsp;

&nbsp;

# Selection of Differencially Peaked Region (DPR)

&nbsp;

## (1) Filtering Condition

- `log2(Fold Change) < -0.5`
- [script/diff-analysis/diff_analysis_for_ChIP_ATAC_DNase.sh](https://github.com/inutano/chip-atlas/blob/master/script/diff-analysis/diff_analysis_for_ChIP_ATAC_DNase.sh)

In [4]:
all_diff_down = all_diff_df.filter(
    pl.col("log2fc") < -0.5
)

display(all_diff_down)

log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str
-1.231767,0.000066,0.000855,"""chr1""",2199175,2199643,"""FCHS_892""","""cp10@FAAP20""",159,0.339744,"{2199409,2199568,""chr1""}",0.317889,"""PLA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-1.86908,2.2899e-8,9.2754e-7,"""chr1""",2208274,2208556,"""FCHS_447710""","""FCHS_447710""",168,0.595745,"{2208337,2208505,""chr1""}",-0.777778,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-1.780242,3.3117e-9,1.7312e-7,"""chr1""",2246149,2246455,"""FCHS_447726""","""FCHS_447726""",241,0.787582,"{2246214,2246601,""chr1""}",0.614679,"""PLA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-1.724787,0.000897,0.007128,"""chr1""",2290632,2291109,"""FCHS_447738""","""FCHS_447738""",56,0.1174,"{2290753,2290809,""chr1""}",0.278195,"""PLA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-2.39651,2.7891e-8,0.000001,"""chr1""",2290632,2291109,"""FCHS_447738""","""FCHS_447738""",116,0.243187,"{2290809,2290925,""chr1""}",0.278195,"""PLA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
-2.598487,2.7698e-8,0.000001,"""chrY""",19077586,19077787,"""FCHS_447247""","""FCHS_447247""",51,0.253731,"{19077498,19077637,""chrY""}",-0.428571,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-2.512691,0.000264,0.002643,"""chrY""",19077163,19077775,"""FCHS_447246""","""cp8@TTTY14""",94,0.153595,"{19077637,19077731,""chrY""}",-0.998313,"""PLA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-2.512691,0.000264,0.002643,"""chrY""",19077586,19077787,"""FCHS_447247""","""FCHS_447247""",94,0.467662,"{19077637,19077731,""chrY""}",-0.428571,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## (2) Filter by Overlap Fraction

- 対象は直前の `log2fc < -0.5` 通過後（`all_diff_down`）
- 同一 `diff_dataset` × `cre_id` で複数 DPR が重なる場合、`overlap_frac_of_fantabio_cre` が最大の行のみ残す
- check: `FCHS_447246` の before / after を表示


In [5]:
# Among downregulated DPRs (log2fc < -0.5), keep max CRE-cover fraction
# per (diff_dataset, cre_id). Ties keep multiple rows.
CHECK_CRE_ID = "FCHS_447246"

fchs_before = all_diff_down.filter(pl.col("cre_id") == CHECK_CRE_ID).sort(
    "diff_dataset",
    "overlap_frac_of_fantabio_cre",
    descending=[False, True],
)
print(f"[before] {CHECK_CRE_ID}: {fchs_before.height} rows")
display(fchs_before)

all_diff_down_filtered = all_diff_down.filter(
    pl.col("overlap_frac_of_fantabio_cre")
    == pl.col("overlap_frac_of_fantabio_cre").max().over("diff_dataset", "cre_id")
)

fchs_after = all_diff_down_filtered.filter(pl.col("cre_id") == CHECK_CRE_ID).sort(
    "diff_dataset",
    "overlap_frac_of_fantabio_cre",
    descending=[False, True],
)
print(
    f"[after]  {CHECK_CRE_ID}: {fchs_after.height} rows "
    f"(all_diff_down {all_diff_down.height} -> {all_diff_down_filtered.height})"
)
display(fchs_after)

[before] FCHS_447246: 3 rows


log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str
-2.598487,2.7698e-8,0.000001,"""chrY""",19077163,19077775,"""FCHS_447246""","""cp8@TTTY14""",139,0.227124,"{19077498,19077637,""chrY""}",-0.998313,"""PLA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-2.512691,0.000264,0.002643,"""chrY""",19077163,19077775,"""FCHS_447246""","""cp8@TTTY14""",94,0.153595,"{19077637,19077731,""chrY""}",-0.998313,"""PLA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-2.288707,1.9434e-7,0.000013,"""chrY""",19077163,19077775,"""FCHS_447246""","""cp8@TTTY14""",139,0.227124,"{19077498,19077637,""chrY""}",-0.998313,"""PLA""","""GSE101731_atac_hff_44C_60min""","""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


[after]  FCHS_447246: 2 rows (all_diff_down 4745 -> 4319)


log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str
-2.598487,2.7698e-8,0.000001,"""chrY""",19077163,19077775,"""FCHS_447246""","""cp8@TTTY14""",139,0.227124,"{19077498,19077637,""chrY""}",-0.998313,"""PLA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-2.288707,1.9434e-7,0.000013,"""chrY""",19077163,19077775,"""FCHS_447246""","""cp8@TTTY14""",139,0.227124,"{19077498,19077637,""chrY""}",-0.998313,"""PLA""","""GSE101731_atac_hff_44C_60min""","""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


&nbsp;

&nbsp;

&nbsp;

## After Filtering

In [6]:
display(all_diff_down_filtered)

log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str
-1.231767,0.000066,0.000855,"""chr1""",2199175,2199643,"""FCHS_892""","""cp10@FAAP20""",159,0.339744,"{2199409,2199568,""chr1""}",0.317889,"""PLA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-1.86908,2.2899e-8,9.2754e-7,"""chr1""",2208274,2208556,"""FCHS_447710""","""FCHS_447710""",168,0.595745,"{2208337,2208505,""chr1""}",-0.777778,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-1.780242,3.3117e-9,1.7312e-7,"""chr1""",2246149,2246455,"""FCHS_447726""","""FCHS_447726""",241,0.787582,"{2246214,2246601,""chr1""}",0.614679,"""PLA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-2.39651,2.7891e-8,0.000001,"""chr1""",2290632,2291109,"""FCHS_447738""","""FCHS_447738""",116,0.243187,"{2290809,2290925,""chr1""}",0.278195,"""PLA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-1.724822,0.000897,0.007128,"""chr1""",2412046,2412449,"""FCHS_1015""","""cp3@PEX10""",78,0.193548,"{2412339,2412417,""chr1""}",0.355212,"""PLA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
-1.724613,0.000034,0.000928,"""chrX""",149540621,149541241,"""FCHS_445688""","""cp5@EOLA1,cp1@EOLA1-DT""",74,0.119355,"{149540922,149540996,""chrX""}",-0.61123,"""PLA""","""GSE101731_atac_hff_44C_60min""","""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-2.598487,2.7698e-8,0.000001,"""chrY""",19077163,19077775,"""FCHS_447246""","""cp8@TTTY14""",139,0.227124,"{19077498,19077637,""chrY""}",-0.998313,"""PLA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-2.512691,0.000264,0.002643,"""chrY""",19077586,19077787,"""FCHS_447247""","""FCHS_447247""",94,0.467662,"{19077637,19077731,""chrY""}",-0.428571,"""ELA""","""GSE101731_atac_hff_44C_120min""","""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


In [7]:
# Aggregate downregulated + max-overlap DPRs to CRE level
all_diff_down_select = all_diff_down_filtered.group_by(
    [
        "cre_chrom",
        "cre_start",
        "cre_end",
        "cre_id",
        "cre_name",
        # "dpr_chrom",
        "directionality",
        "fantabio_cre_class"
    ]
).agg(
    pl.col("overlap_frac_of_fantabio_cre").alias("overlap_frac_of_fantabio_cre"),
    pl.col("dpr_interval").alias("dpr_interval"),
    pl.col("log2fc").alias("log2fc"),
    pl.col("p_value").alias("p_value"),
    pl.col("q_value").alias("q_value"),
    pl.col("diff_dataset").unique().alias("diff_dataset"),
    pl.col("diff_dataset").unique().count().alias("diff_dataset_count"),
    pl.col("source_bed_file").alias("source_bed_file"),
).sort(
    "cre_chrom",
    "cre_start",
    "cre_end",
    descending=False
)

pl.Config.set_tbl_rows(6)
display(all_diff_down_select)

cre_chrom,cre_start,cre_end,cre_id,cre_name,directionality,fantabio_cre_class,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,str,str,f64,str,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""chr1""",2199175,2199643,"""FCHS_892""","""cp10@FAAP20""",0.317889,"""PLA""","[0.339744, 0.339744]","[{2199409,2199568,""chr1""}, {2199409,2199568,""chr1""}]","[-1.231767, -1.275303]","[0.000066, 0.000015]","[0.000855, 0.000479]","[""GSE101731_atac_hff_44C_120min"", ""GSE101731_atac_hff_44C_60min""]",2,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""chr1""",2208274,2208556,"""FCHS_447710""","""FCHS_447710""",-0.777778,"""ELA""",[0.595745],"[{2208337,2208505,""chr1""}]",[-1.86908],[2.2899e-8],[9.2754e-7],"[""GSE101731_atac_hff_44C_120min""]",1,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""chr1""",2246149,2246455,"""FCHS_447726""","""FCHS_447726""",0.614679,"""PLA""","[0.787582, 0.787582]","[{2246214,2246601,""chr1""}, {2246214,2246601,""chr1""}]","[-1.780242, -1.719544]","[3.3117e-9, 5.8502e-9]","[1.7312e-7, 6.3413e-7]","[""GSE101731_atac_hff_44C_120min"", ""GSE101731_atac_hff_44C_60min""]",2,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrX""",155613048,155613375,"""FCHS_446859""","""cp1@TMLHE""",-0.947368,"""ELA""",[0.042813],"[{155612829,155613062,""chrX""}]",[-1.889519],[0.000005],[0.000098],"[""GSE101731_atac_hff_44C_120min""]",1,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""chrY""",19077163,19077775,"""FCHS_447246""","""cp8@TTTY14""",-0.998313,"""PLA""","[0.227124, 0.227124]","[{19077498,19077637,""chrY""}, {19077498,19077637,""chrY""}]","[-2.598487, -2.288707]","[2.7698e-8, 1.9434e-7]","[0.000001, 0.000013]","[""GSE101731_atac_hff_44C_120min"", ""GSE101731_atac_hff_44C_60min""]",2,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""chrY""",19077586,19077787,"""FCHS_447247""","""FCHS_447247""",-0.428571,"""ELA""","[0.467662, 0.253731]","[{19077637,19077731,""chrY""}, {19077498,19077637,""chrY""}]","[-2.512691, -2.288707]","[0.000264, 1.9434e-7]","[0.002643, 0.000013]","[""GSE101731_atac_hff_44C_120min"", ""GSE101731_atac_hff_44C_60min""]",2,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"


In [8]:
all_diff_down_select.write_parquet(
    "../../out/ChIP-Atlas_diff/diff_analysis_filter_parquet/only_cre/fantabio_cre_list_diff_analysis_atac_down.parquet"
)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Intergration (downregulated transcripts)

In [9]:
# down5p transcript lists + CRE
fantabio_HN2_down5p_transcripts_cre = pl.read_parquet(
    "../../out/fantabio/parquet_file/bottom5p_transcript_fantabio_v261001.parquet"
).join(
    all_diff_down_select,
    on=["cre_id", "cre_chrom", "cre_name", "directionality", "fantabio_cre_class"],
    how="inner",
    coalesce=True
)

fantabio_HN2_down5p_transcripts_cre.write_parquet(
    "../../out/ChIP-Atlas_diff/diff_analysis_filter_parquet/transcript_cre/fantabio_HN2_bottom5p_v261002_diff_analysis_atac.parquet"
)

print(f"down5p_fantabio transcript count: {fantabio_HN2_down5p_transcripts_cre.select('transcript_id').n_unique()}")
print(f"down5p_fantabio transcript-CRE pairs count: {fantabio_HN2_down5p_transcripts_cre.select(['transcript_id', 'cre_id']).n_unique()}")
pl.Config.set_tbl_rows(4)
display(fantabio_HN2_down5p_transcripts_cre)

down5p_fantabio transcript count: 1380
down5p_fantabio transcript-CRE pairs count: 1925


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000489344.5""",69,117,220,-48,44,62,145,-18,"""RBM3""","""HGNC:9900""","""protein_coding_CDS_not_defined""","""ENSG00000102317.19""",[],"""+""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA binding motif protein 3""]",1,"""FCHS_439110""","""chrX""",48574136,"""48574672""","""cp3@RBM3""","[""ENST00000472897.5"", ""ENST00000488216.5"", … ""AY203954""]","""0""",0,"[{""EH38E2754036"",""PLS""}, {""EH38E3933736"",""PLS""}]","[{""NRF1"",4624,[{""SRX2424029"",4624}]}, {""MYC"",3976,[{""SRX21459788"",3976}, {""SRX21459787"",3815}, … {""SRX17214576"",804}]}, … {""SUMO2"",501,[{""SRX895419"",501}]}]",null,"[{""hg19::chrX:48432865..48432880,+;hg_196262.1"",""p2@RBM3""}, {""hg19::chrX:48432892..48432916,+;hg_196263.1"",""p1@RBM3""}]",null,0.999458,"""PLA""","""204,0,0""","""red""",48574135,48574672,[0.428305],"[{48574269,48574499,""chrX""}]",[-1.268999],[0.000932],[0.00734],"[""GSE101731_atac_hff_44C_120min""]",1,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000870205.1""",77,125,204,-48,50,70,131,-20,"""ATPAF1""","""HGNC:18803""","""protein_coding""","""ENSG00000123472.14""","[""basic"", ""TAGENE""]","""-""",2,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""","[""DanaFarber""]",1,"""FCHS_13476""","""chr1""",46668149,"""46669135""","""cp1@ATPAF1,cp3@TEX38""","[""ENST00000870202.1"", ""ENST00000574428.6"", … ""EU831580""]","""0""",0,"[{""EH38E1344898"",""pELS""}, {""EH38E1344900"",""pELS""}, {""EH38E2809485"",""PLS""}]","[{""ZBTB11"",2998,[{""SRX16495400"",2998}, {""SRX3322700"",2883}, … {""SRX2424559"",572}]}, {""E2F1"",2958,[{""SRX150556"",2958}, {""SRX992662"",1575}, … {""SRX334262"",579}]}, … {""IKZF1"",500,[{""SRX3322408"",500}]}]","[{""HS_00005648"",""Fetal Brain Female"",""Brain"",""Brain"",""SVM""}, {""HS_00005649"",""HepG2"",""Liver"",""Liver"",""SVM""}, {""HS_03038100"",""K562"",""Blood"",""Blood"",""High throughput""}]","[{""hg19::chr1:47133988..47134051,-;hg_12192.1"",""p1@ATPAF1""}, {""hg19::chr1:47134067..47134084,-;hg_12193.1"",""p4@ATPAF1""}, … {""hg19::chr1:47134132..47134135,-;hg_12196.1"",""p6@ATPAF1""}]",null,-0.99993,"""PLA""","""0,0,204""","""blue""",46668148,46669135,[0.165147],"[{46668266,46668429,""chr1""}]",[-1.077844],[0.001062],[0.008142],"[""GSE101731_atac_hff_44C_120min""]",1,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000678149.1""",10,217,179,-207,2,138,111,-136,"""SKP2""","""HGNC:10901""","""retained_intron""","""ENSG00000145604.18""","[""RNA_Seq_supported_only""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported only""]",1,"""FCHS_327453""","""chr5""",36151719,"""36152301""","""cp1@LMBRD2,cp6@SKP2""","[""ENST00000296603.5"", ""ENST00000955895.1"", … ""AY029177""]","""0""",0,"[{""EH38E3632338"",""PLS""}, {""EH38E3632339"",""PLS""}, {

&nbsp;

## Protein-coding Transcripts

In [11]:
# (check whether the protein-coding transcripts - CREs are heat stress-responsive related transcripts)
fantabio_HN2_down5p_transcripts_cre_pc = fantabio_HN2_down5p_transcripts_cre.filter(
    pl.col("transcript_type") == "protein_coding"
)

down5p_pc_list = fantabio_HN2_down5p_transcripts_cre_pc.select("transcript_id").unique().write_csv(
    "../../out/metascape/transcript_list/down5p/HN2_down5p_fantabio_check_diff_analysis_atac_pc_v261002.tsv",
    separator="\t",
    include_header=False
)

print(f"down5p_fantabio protein-coding transcript count: {fantabio_HN2_down5p_transcripts_cre_pc.select("transcript_id").n_unique()}")
print(f"down5p_fantabio protein-coding transcript-CRE pairs count: {fantabio_HN2_down5p_transcripts_cre_pc.select(['transcript_id', 'cre_id']).n_unique()}")
display(fantabio_HN2_down5p_transcripts_cre_pc)

down5p_fantabio protein-coding transcript count: 772
down5p_fantabio protein-coding transcript-CRE pairs count: 1059


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000870205.1""",77,125,204,-48,50,70,131,-20,"""ATPAF1""","""HGNC:18803""","""protein_coding""","""ENSG00000123472.14""","[""basic"", ""TAGENE""]","""-""",2,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""","[""DanaFarber""]",1,"""FCHS_13476""","""chr1""",46668149,"""46669135""","""cp1@ATPAF1,cp3@TEX38""","[""ENST00000870202.1"", ""ENST00000574428.6"", … ""EU831580""]","""0""",0,"[{""EH38E1344898"",""pELS""}, {""EH38E1344900"",""pELS""}, {""EH38E2809485"",""PLS""}]","[{""ZBTB11"",2998,[{""SRX16495400"",2998}, {""SRX3322700"",2883}, … {""SRX2424559"",572}]}, {""E2F1"",2958,[{""SRX150556"",2958}, {""SRX992662"",1575}, … {""SRX334262"",579}]}, … {""IKZF1"",500,[{""SRX3322408"",500}]}]","[{""HS_00005648"",""Fetal Brain Female"",""Brain"",""Brain"",""SVM""}, {""HS_00005649"",""HepG2"",""Liver"",""Liver"",""SVM""}, {""HS_03038100"",""K562"",""Blood"",""Blood"",""High throughput""}]","[{""hg19::chr1:47133988..47134051,-;hg_12192.1"",""p1@ATPAF1""}, {""hg19::chr1:47134067..47134084,-;hg_12193.1"",""p4@ATPAF1""}, … {""hg19::chr1:47134132..47134135,-;hg_12196.1"",""p6@ATPAF1""}]",null,-0.99993,"""PLA""","""0,0,204""","""blue""",46668148,46669135,[0.165147],"[{46668266,46668429,""chr1""}]",[-1.077844],[0.001062],[0.008142],"[""GSE101731_atac_hff_44C_120min""]",1,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000676271.1""",37,85,284,-48,28,51,172,-23,"""KIF2A""","""HGNC:6318""","""protein_coding""","""ENSG00000068796.20""","[""basic"", ""appris_alternative_1"", ""CCDS""]","""+""",3,null,null,"[""polya_site"", ""polya_signal""]","""ensembl_havana_gene_homo_sapiens""","[""align_id:3708271|asmbl_198436""]",1,"""FCHS_330312""","""chr5""",62305653,"""62306109""","""cp6@KIF2A""","[""ENST00000401507.7""]","""53""",343,"[{""EH38E2376862"",""PLS""}, {""EH38E3639620"",""pELS""}]","[{""NRF1"",5369,[{""SRX2424029"",5369}, {""SRX2424732"",5287}, … {""SRX2424028"",2047}]}, {""E2F1"",3714,[{""SRX150556"",3714}, {""SRX992662"",2008}, … {""SRX334247"",523}]}, … {""RAD51"",502,[{""SRX2424086"",502}]}]",null,"[{""hg19::chr5:61601800..61601812,+;hg_56639.1"",""p12@KIF2A""}]",null,0.903922,"""PLA""","""204,0,0""","""red""",62305652,62306109,[0.223195],"[{62305904,62306006,""chr5""}]",[-1.135536],[0.000281],[0.002781],"[""GSE101731_atac_hff_44C_120min""]",1,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000527436.5""",12,200,194,-188,8,142,101,-134,"""ZSCAN9""","""HGNC:12984""","""protein_coding""","""ENSG00000137185.14""","[""basic"", ""CCDS""]","""+""",2,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_353038""","""chr6""",28225024,"""28225472""","""cp2@ZSCAN9""","[""ENST00000531981.5"", ""ENST00000425468.6"", … ""U62392""]","""0""",0,"[{""EH38E2457123"",""PLS""}]","[{""ZBTB11"",3776,[{""SRX15898392"",3776}, {""SRX2424559

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## lncRNA

In [12]:
fantabio_HN2_down5p_transcripts_cre_lncrna = fantabio_HN2_down5p_transcripts_cre.filter(
    pl.col("transcript_type") == "lncRNA"
).sort(
    pl.col([
        "HN2",
        "transcript_id"
    ]),
    descending=True
)

down5p_lncrna_list = fantabio_HN2_down5p_transcripts_cre_lncrna.select("transcript_id").unique().write_csv(
    "../../out/metascape/transcript_list/down5p/HN2_down5p_fantabio_check_diff_analysis_atac_lncrna_v261002.tsv",
    separator="\t",
    include_header=False
)

pl.Config.set_tbl_rows(4)
print(f"down5p_fantabio lncRNA transcript count: {fantabio_HN2_down5p_transcripts_cre_lncrna.select("transcript_id").n_unique()}")
print(f"down5p_fantabio lncRNA transcript-CRE pairs count: {fantabio_HN2_down5p_transcripts_cre_lncrna.select(['transcript_id', 'cre_id']).n_unique()}")
display(fantabio_HN2_down5p_transcripts_cre_lncrna)

down5p_fantabio lncRNA transcript count: 229
down5p_fantabio lncRNA transcript-CRE pairs count: 346


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000837435.1""",50,98,258,-48,29,70,152,-41,"""OR2A1-AS1""","""HGNC:49168""","""lncRNA""","""ENSG00000244479.8""","[""TAGENE""]","""-""",1,"""URS0002A19989""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_576620""","""chr7""",144355020,"""144355395""","""cp1@ARHGEF5,cp2@OR2A1-AS1""","[""ENST00000837389.1"", ""ENST00000837084.1"", … ""AK160365""]","""0""",0,null,"[{""AR"",2903,[{""SRX11646095"",2903}, {""SRX11646097"",2154}, … {""SRX12971688"",504}]}, {""STAT3"",2653,[{""SRX5323860"",2653}, {""SRX5323888"",2329}, … {""SRX10119380"",551}]}, … {""WAS"",509,[{""SRX5371250"",509}]}]",null,"[{""hg19::chr7:144052487..144052509,+;hg_81808.1"",""p2@ARHGEF5""}]",null,-1.0,"""ELA""","""255,202,74""","""yellow""",144355019,144355395,"[0.510638, 0.510638]","[{144355145,144355337,""chr7""}, {144355145,144355337,""chr7""}]","[-1.479999, -1.378654]","[0.000068, 0.000036]","[0.000877, 0.000959]","[""GSE101731_atac_hff_44C_120min"", ""GSE101731_atac_hff_44C_60min""]",2,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000836881.1""",77,125,204,-48,38,82,131,-44,null,null,"""lncRNA""","""ENSG00000287780.2""","[""TAGENE""]","""-""",1,"""URS0002A51F68""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_283071""","""chr3""",39383211,"""39383568""","""cp4@SLC25A38""","[""ENST00000885742.1"", ""ENST00000929325.1"", … ""AK225388""]","""0""",-447,"[{""EH38E2193804"",""pELS""}, {""EH38E3507039"",""PLS""}]","[{""CTCF"",3018,[{""SRX10975999"",3018}, {""SRX10976000"",2442}, … {""SRX2636092"",501}]}, {""NR2C2"",2933,[{""SRX3322933"",2933}, {""SRX10478064"",2278}, … {""SRX150675"",586}]}, … {""BAHCC1"",507,[{""SRX8449940"",507}]}]","[{""HS_00040449"",""HepG2"",""Liver"",""Liver"",""correlation""}, {""HS_00040450"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""correlation""}, … {""HS_00040457"",""Foreskin Melanocyte Primary Cells skin01"",""Prepuce"",""Skin"",""correlation""}]","[{""hg19::chr3:39424828..39424918,+;hg_36131.1"",""p1@SLC25A38""}]",null,0.982632,"""PLA""","""204,0,0""","""red""",39383210,39383568,[0.215084],"[{39383365,39383442,""chr3""}]",[-1.560151],[0.000075],[0.000948],"[""GSE101731_atac_hff_44C_120min""]",1,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000564063.1""",22,193,191,-171,13,134,104,-121,null,null,"""lncRNA""","""ENSG00000260879.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,"""URS0000530217""","""ENSEMBL_GENCODE""","[""polya_site"", ""polya_signal""]","""havana_homo_sapiens""",null,0,"""FCHS_20586""","""chr1""",108199982,"""108200728""","""cp3@SLC25A24""","[""ENST00000565488.6"", ""ENST00000264128.13"", … ""EU832606""]","""0""",-56,"[{""EH38E2829026"",""PLS""

&nbsp;

&nbsp;

&nbsp;

## Other Transcripts

In [13]:
fantabio_HN2_down5p_transcripts_cre_other = fantabio_HN2_down5p_transcripts_cre.filter(
    (pl.col("transcript_type") != "lncRNA") & (pl.col("transcript_type") != "protein_coding")
).sort(
    pl.col("HN2"),
    descending=True
)

down5p_other_list = fantabio_HN2_down5p_transcripts_cre_other.select("transcript_id").unique().write_csv(
    "../../out/metascape/transcript_list/down5p/HN2_down5p_fantabio_check_diff_analysis_atac_other_v261002.tsv",
    separator="\t",
    include_header=False
)

pl.Config.set_tbl_rows(10)
print(fantabio_HN2_down5p_transcripts_cre_other.select("transcript_id").n_unique())
print(fantabio_HN2_down5p_transcripts_cre_other.group_by("transcript_type").agg(pl.col("transcript_id").n_unique().alias("transcript_type_count")))
display(fantabio_HN2_down5p_transcripts_cre_other)

379
shape: (6, 2)
┌────────────────────────────────────┬───────────────────────┐
│ transcript_type                    ┆ transcript_type_count │
│ ---                                ┆ ---                   │
│ str                                ┆ u32                   │
╞════════════════════════════════════╪═══════════════════════╡
│ non_stop_decay                     ┆ 2                     │
│ protein_coding_CDS_not_defined     ┆ 78                    │
│ nonsense_mediated_decay            ┆ 91                    │
│ TEC                                ┆ 6                     │
│ transcribed_unprocessed_pseudogene ┆ 1                     │
│ retained_intron                    ┆ 201                   │
└────────────────────────────────────┴───────────────────────┘


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000489344.5""",69,117,220,-48,44,62,145,-18,"""RBM3""","""HGNC:9900""","""protein_coding_CDS_not_defined""","""ENSG00000102317.19""",[],"""+""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA binding motif protein 3""]",1,"""FCHS_439110""","""chrX""",48574136,"""48574672""","""cp3@RBM3""","[""ENST00000472897.5"", ""ENST00000488216.5"", … ""AY203954""]","""0""",0,"[{""EH38E2754036"",""PLS""}, {""EH38E3933736"",""PLS""}]","[{""NRF1"",4624,[{""SRX2424029"",4624}]}, {""MYC"",3976,[{""SRX21459788"",3976}, {""SRX21459787"",3815}, … {""SRX17214576"",804}]}, … {""SUMO2"",501,[{""SRX895419"",501}]}]",null,"[{""hg19::chrX:48432865..48432880,+;hg_196262.1"",""p2@RBM3""}, {""hg19::chrX:48432892..48432916,+;hg_196263.1"",""p1@RBM3""}]",null,0.999458,"""PLA""","""204,0,0""","""red""",48574135,48574672,[0.428305],"[{48574269,48574499,""chrX""}]",[-1.268999],[0.000932],[0.00734],"[""GSE101731_atac_hff_44C_120min""]",1,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000357434.3""",27,75,304,-48,14,38,199,-24,"""CEP350""","""HGNC:24238""","""retained_intron""","""ENSG00000135837.19""","[""RNA_Seq_supported_only""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported only""]",1,"""FCHS_30777""","""chr1""",179954331,"""179955003""","""cp5@CEP350""","[""ENST00000706241.1"", ""ENST00000490047.2"", … ""AF287356""]","""0""",0,"[{""EH38E1401725"",""PLS""}, {""EH38E1401726"",""PLS""}, {""EH38E2850863"",""PLS""}]","[{""ZNF143"",3588,[{""SRX1510972"",3588}, {""SRX1510970"",3115}, … {""SRX14835717"",531}]}, {""BANP"",3442,[{""SRX8873455"",3442}, {""SRX8873456"",2959}, … {""SRX8873457"",2478}]}, … {""LIN9"",500,[{""SRX4213897"",500}]}]","[{""HS_00014869"",""H1 Derived Neuronal Progenitor Cultured Cells"",""Brain"",""Brain"",""SVM""}, {""HS_00014870"",""Foreskin Melanocyte Primary Cells skin01"",""Prepuce"",""Skin"",""SVM""}, … {""HS_00014875"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}]","[{""hg19::chr1:179923716..179923724,+;hg_6939.1"",""p2@CEP350""}, {""hg19::chr1:179923826..179923847,-;hg_16098.1"",""hg_16098.1""}, {""hg19::chr1:179923875..179923954,+;hg_6940.1"",""p1@CEP350""}]",null,0.991337,"""PLA""","""204,0,0""","""red""",179954330,179955003,"[0.23477, 0.499257]","[{179954482,179954640,""chr1""}, {179954482,179954818,""chr1""}]","[-1.770068, -0.821336]","[1.0407e-11, 0.000003]","[8.7537e-10, 0.000136]","[""GSE101731_atac_hff_44C_120min"", ""GSE101731_atac_hff_44C_60min""]",2,"[""GSE101731_atac_hff_44C_120min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""GSE101731_atac_hff_44C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000357434.3""",27,75,304,-48,14,38,199,-24,"""CEP350""","""HGNC:24238""","""retained_intron""","""ENSG00000135837.19""","[""RNA_Seq_supported_only""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported only""]",1,"""FCHS_30778""","""chr1"